(fin-edu:history:us-equity)=
# USA equity market

In 1926 the Stardard Statistics Company developed a 90-stock index. On March 4, 1957, Standard and Poor's - the name of the company after the merging of the SSC and the Poor's Publishing - expanded the index to the current number of 500 companies, renamed the S&P500 Stock Composite Index.

On August 31, 1976, The Vanguard Group launched the first mutual fund to retail investors that tracked the index.

**Table of contents.**
- **Return** over different time frames, both in nominal and real value
- **Sectors**


In [75]:
import numpy as np

import pandas as pd
# from IPython.display import display  # import not required, maybe

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly.io as pio
pio.renderers.default = "iframe"

(fin-edu:history:us-equity:return)=
## Return

- Time history of the total return index
- Return over different time spans: probability function and statistics, composition of returns
- Volatility: drag, volatility clusters,...


(fin-edu:history:us-equity:return:total)=
### Total Return - Shiller data

Shiller data, from 1871 to 2025

In [82]:
#> Import Shiller data
shiller_filen =  "./../../code/data/shiller/ie_data.xls"
df = pd.read_excel(shiller_filen, sheet_name="Data", skiprows=7, engine="xlrd")

#> Clean data
#> Remove non-data last row
df = df.iloc[:-1].copy()

#> Convert Data to proper datetime type
df["Date"] = df["Date"].map(lambda x: f"{x:.2f}")
df["Date"] = pd.to_datetime( df["Date"], format="%Y.%m")

#> Column names
# P: S&P Comp. P
# Price: Real Price
# Price.1: Real Total Return Price
# print(df.columns)

# display(df.head())
# display(df.tail())

,Date,P,D,E,CPI,Fraction,Rate GS10,Price,Dividend,Price.1,...,CAPE,Unnamed: 13,TR CAPE,Unnamed: 15,Yield,Returns,Returns.1,Real Return,Real Return.1,Returns.2
0,1871-01-01,4.44,0.26,0.4,12.464061,1871.041667,5.32,114.692610,6.716234,114.692610,...,NaN,NaN,NaN,NaN,NaN,1.004177,1.000000,0.130609,0.092504,0.038106
1,1871-02-01,4.5,0.26,0.4,12.844641,1871.125000,5.323333,112.798304,6.517235,113.341406,...,NaN,NaN,NaN,NaN,NaN,1.004180,0.974424,0.130858,0.094635,0.036224
2,1871-03-01,4.61,0.26,0.4,13.034972,1871.208333,5.326667,113.868306,6.422074,114.954311,...,NaN,NaN,NaN,NaN,NaN,1.004183,0.964209,0.130951,0.096186,0.034765
3,1871-04-01,4.74,0.26,0.4,12.559226,1871.291667,5.33,121.514327,6.665343,123.233997,...,NaN,NaN,NaN,NaN,NaN,1.004185,1.004919,0.122056,0.090972,0.031084
4,1871-05-01,4.86,0.26,0.4,12.273812,1871.375000,5.333333,127.487866,6.820339,129.868479,...,NaN,NaN,NaN,NaN,NaN,1.004188,1.032591,0.122638,0.089488,0.033150


In [85]:
#> S&P 500 index (even if the S&P500 was created in 1957; blended with some other index?)
#> Plots: (2,1) subplot: density and cumulative probability
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=True,
    vertical_spacing=0.1,
    # subplot_titles=("Monthly Returns Probability Density", "Cumulative Probability")
)

#> S&P500 index
fig.add_trace(go.Scatter(x=df["Date"], y=df["P"], name="S&P 500"),
    row=1, col=1
)
fig.add_trace(go.Scatter(x=df["Date"], y=df["CPI"], name="CPI"),
    row=1, col=2
)
fig.add_trace(go.Scatter(x=df["Date"], y=df["Price"], name="Real S&P500 Index"),
    row=2, col=1
)
fig.add_trace(go.Scatter(x=df["Date"], y=df["Price.1"], name="Real Total Return"),
    row=2, col=2
)

fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_yaxes(type="log", row=2, col=1)
fig.update_yaxes(type="log", row=2, col=2)

fig.show()



In [81]:
#> Monthly returns
df["return_month"] = df["Price.1"].pct_change()
mu, sig2 = df["return_month"].mean(), df["return_month"].var()

#> Statistics
print(df["return_month"].describe())
print(f"Skewness: {df['return_month'].skew()}")
print(f"Kurtosis: {df['return_month'].kurtosis()}")

#> Plots: (2,1) subplot: density and cumulative probability
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    # subplot_titles=("Monthly Returns Probability Density", "Cumulative Probability")
)

# Top: probability density histogram
fig.add_trace(
    go.Histogram(
        x=df["return_month"].dropna(),
        # nbinsx=50,
        histnorm='probability',  # normalize
        name='Density',
        # marker_color='steelblue',
        opacity=0.7
    ),
    row=1, col=1
)

# x_plot = np.linspace(min(df['return_month'].dropna()),max(df['return_month'].dropna()), 100)
# print(x_plot)
# fig.add_trace(
#     go.Scatter(
#         x=x_plot,
#         y=np.exp(-(x_plot-mu)**2/(2.*sig2)) / np.sqrt(2*np.pi*sig2)
#     ),
#     row=1, col=1
# )

# Bottom: cumulative histogram
fig.add_trace(
    go.Histogram(
        x=df["return_month"].dropna(),
        # nbinsx=50,
        cumulative_enabled=True,       # cumulative
        histnorm='probability',        # normalize cumulative to 1
        name='Cumulative',
        # marker_color='orange',
        opacity=0.7
    ),
    row=2, col=1
)

# Layout tweaks
fig.update_layout(
    title="S&P 500 Monthly Returns: Probability Density & Cumulative Probability",
    width=800, height=600,
    xaxis_title="Monthly Return",
    yaxis_title="Density",
    xaxis2_title="Monthly Return",
    yaxis2_title="Cumulative Probability",
    template="plotly_white",
    bargap=0.1,
    showlegend=False
)

fig.show()

count    1854.000000
mean        0.006515
std         0.040827
min        -0.261879
25%        -0.013347
50%         0.009404
75%         0.029377
max         0.524294
Name: return_month, dtype: float64
Skewness: 0.5564109460860193
Kurtosis: 17.948214818518505


(fin-edu:history:us-equity:sectors)=
## Sectors

**Message:** market composition changes - rise and fall of dominant sectors; diversification can disappear as market concentration in one or few sectors increases; survivorship and return bias.

**References.**
- Kenneth Franch Data Library: industry portfolios by market cap, long history, clean and ideal for education, https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
- Robert Shiller data
  - http://www.econ.yale.edu/~shiller/data.htm
  - https://shillerdata.com/
- Wikipedia for top companies: scratch largest companies by mkt cap every 5-10 year and then show the evolution of the market cap